# LSTM

In [5]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

import sys
import os

sys.path.append(os.path.abspath('..'))

In [7]:
df = pd.read_csv('../../Data/aluminium_pre_inputs.csv')

In [8]:
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length - 1):
        x = data[i:(i + seq_length)]
        y = data[i + seq_length]
        xs.append(x)
        ys.append(y)

    return np.array(xs), np.array(ys)

In [14]:
X, y = create_sequences(df['al_lme_prices_log_returns'], 5)

In [15]:
train_size = int(len(y)*0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

X_train = torch.from_numpy(X_train).float()
X_test = torch.from_numpy(X_test).float()
y_train = torch.from_numpy(y_train).float()
y_test = torch.from_numpy(y_test).float()

In [16]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.lstm(x, (h0, c0))
        out = self.linear(out[:, -1, :])  # take the last output
        return out

In [19]:
input_size = 1
hidden_size = 50
num_layers = 2
learning_rate = 0.01
EPOCHS = 100

model = LSTMModel()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(EPOCHS):
    model.train()
    output = model(X_train.unsqueeze(-1))
    loss = criterion(output, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}], Loss: {loss.item():.6f}')

c:\Users\PanSt\Desktop\UCL\T3\Dissertation\Commodity-Optionality-Pricing-UCL-Dissertation\.venv\Lib\site-packages\torch\nn\modules\loss.py:610: UserWarning: Using a target size (torch.Size([1951])) that is different to the input size (torch.Size([1951, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [10/100], Loss: 0.000393
Epoch [20/100], Loss: 0.000525
Epoch [30/100], Loss: 0.000242
Epoch [40/100], Loss: 0.000180
Epoch [50/100], Loss: 0.000190
Epoch [60/100], Loss: 0.000184
Epoch [70/100], Loss: 0.000180
Epoch [80/100], Loss: 0.000180
Epoch [90/100], Loss: 0.000180
Epoch [100/100], Loss: 0.000180


In [21]:
with torch.no_grad():
    test_outputs = model(X_test.unsqueeze(-1))
    test_loss = criterion(test_outputs, y_test)
    print(f'Test Loss: {test_loss.item():.4f}')

Test Loss: 0.0002


c:\Users\PanSt\Desktop\UCL\T3\Dissertation\Commodity-Optionality-Pricing-UCL-Dissertation\.venv\Lib\site-packages\torch\nn\modules\loss.py:610: UserWarning: Using a target size (torch.Size([488])) that is different to the input size (torch.Size([488, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [22]:
def lstm_predict(model, input_data):
    input_data = torch.tensor(input_data).float().unsqueeze(0).unsqueeze(-1)
    with torch.no_grad():
        prediction = model(input_data).squeeze().item()
    return prediction